# Phase 2B — Machine Learning Training, Evaluation & Explainability

**Project:** Airline Delay Prediction & Operations Analytics  
**Scope:** Pre-departure delay classification ($P(\text{arrival delay} \ge 15\text{ min})$), chronological out-of-time validation, multi-model benchmarking (Baseline, Logistic Regression, Random Forest, XGBoost), threshold sensitivity analysis, probability calibration, and SHAP explainability.  
**Dataset:** Validated ML feature dataset (`data/processed/flights_features.parquet`).  

> **DEVELOPMENT DATASET LIMITATION NOTICE:**  
> This notebook runs on the development dataset sample (481 flights from January 1–10, 2024). Results verify pipeline execution, anti-leakage invariants, and explainability architecture. Production deployment requires training on the full BTS dataset.

In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set project root
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.utils.config import get_config
from src.models.split import split_dataset_chronologically
from src.models.train import (
    determine_feature_groups,
    build_preprocessor,
    train_models,
    select_best_model_on_validation,
    extract_feature_importances,
    compute_shap_explainability,
    save_serialized_model,
)
from src.models.evaluate import (
    calculate_classification_metrics,
    compare_models,
    evaluate_thresholds,
    compute_calibration_curve,
    plot_confusion_matrix,
    plot_roc_curves,
    plot_pr_curves,
    plot_calibration_curves,
    plot_feature_importance,
)
from src.models.predict import predict_delay_probability, load_model

config = get_config()
print('Libraries and modular components imported successfully.')

## Section 1: Load Dataset

Load the engineered pre-departure features generated in Phase 2A (`flights_features.parquet`).

In [2]:
features_path = config.data_processed_dir / 'flights_features.parquet'
df = pd.read_parquet(features_path)
print(f'Loaded ML feature dataset: {df.shape[0]} records, {df.shape[1]} columns')
df.head()

## Section 2: Dataset Validation & Leakage Audit

Verify column schemas, missing value patterns, and guarantee that no post-flight outcome features enter the dataset.

In [3]:
from src.features.feature_engineering import assert_no_target_leakage

assert 'delay_target' in df.columns, 'Missing delay_target!'
assert_no_target_leakage(df, config=config)
print('LEAKAGE AUDIT: PASS - Feature matrix contains 0 post-flight leakage variables.')

# Missing value summary
missing = df.isnull().sum()
missing_cols = missing[missing > 0]
print(f'Columns with missing values:\n{missing_cols}')

## Section 3: Target Distribution

Examine the class distribution of `delay_target` (1 = delayed $\ge 15$ min, 0 = on-time).

In [4]:
target_counts = df['delay_target'].value_counts()
target_pct = df['delay_target'].value_counts(normalize=True) * 100

print(f'On-Time Flights (0) : {target_counts.get(0, 0):,} ({target_pct.get(0, 0):.2f}%)')
print(f'Delayed Flights (1) : {target_counts.get(1, 0):,} ({target_pct.get(1, 0):.2f}%)')

fig, ax = plt.subplots(figsize=(5, 4))
bars = ax.bar(['On-Time (0)', 'Delayed (1)'], target_counts.values, color=['#2b5c8f', '#d95f02'], alpha=0.85)
ax.set_ylabel('Number of Flights')
ax.set_title('Target Distribution (Development Sample)', fontweight='bold')
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5, f'{int(bar.get_height())}', ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

## Section 4: Chronological Out-of-Time Split

Split the dataset chronologically: Past (70% Train) $\to$ Intermediate (15% Validation) $\to$ Future (15% Test).

In [5]:
split_res = split_dataset_chronologically(
    df=df,
    date_col='flight_date',
    time_col='scheduled_dep_time',
    target_col='delay_target',
    train_pct=config.train_ratio,
    val_pct=config.val_ratio,
    config=config,
)
X_train, y_train = split_res.X_train, split_res.y_train
X_val, y_val = split_res.X_val, split_res.y_val
X_test, y_test = split_res.X_test, split_res.y_test
meta = split_res.split_summary

print(f'Train: {meta["train_count"]} ({meta["train_date_range"]}) — Delay Rate: {meta["train_delay_rate"]}%')
print(f'Val  : {meta["val_count"]} ({meta["val_date_range"]}) — Delay Rate: {meta["val_delay_rate"]}%')
print(f'Test : {meta["test_count"]} ({meta["test_date_range"]}) — Delay Rate: {meta["test_delay_rate"]}%')
print(f'Temporal Ordering Verified: {meta["train_date_range"][1]} <= {meta["val_date_range"][0]} <= {meta["test_date_range"][0]}')

## Section 5: Preprocessing & Route Feature Analysis

Dynamically inspect numerical and categorical feature groups, analyze route cardinality, and construct leakage-free `ColumnTransformer` pipelines fitted strictly on training data.

In [6]:
num_cols, cat_cols, route_meta = determine_feature_groups(df, target_col='delay_target', config=config)
print(f'Numerical features ({len(num_cols)}): {num_cols[:6]} ...')
print(f'Categorical features ({len(cat_cols)}): {cat_cols}')
print(f'Route decision: {route_meta.get("decision")} — {route_meta.get("reason")}')

preprocessor = build_preprocessor(num_cols, cat_cols, scale_numeric=False)
print('ColumnTransformer preprocessor initialized.')

## Section 6: Baseline Model

Establish a majority-class naive baseline (always predicting on-time) to establish the minimum benchmark.

In [7]:
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import Pipeline

dummy_pipe = Pipeline([
    ('preprocessor', build_preprocessor(num_cols, cat_cols, scale_numeric=False)),
    ('classifier', DummyClassifier(strategy='most_frequent')),
])
dummy_pipe.fit(X_train, y_train)
dummy_probs = dummy_pipe.predict_proba(X_val)[:, 1]
dummy_metrics = calculate_classification_metrics(y_val, dummy_probs, threshold=0.50)
print('Majority Baseline Validation Metrics:', dummy_metrics)

## Section 7: Logistic Regression

Train an interpretable baseline model with numerical standardization and class-weighted cost balancing.

In [8]:
from sklearn.linear_model import LogisticRegression

lr_pipe = Pipeline([
    ('preprocessor', build_preprocessor(num_cols, cat_cols, scale_numeric=True)),
    ('classifier', LogisticRegression(C=config.lr_c, class_weight='balanced', random_state=config.random_seed, max_iter=1000)),
])
lr_pipe.fit(X_train, y_train)
lr_probs = lr_pipe.predict_proba(X_val)[:, 1]
lr_metrics = calculate_classification_metrics(y_val, lr_probs, threshold=0.50)
print('Logistic Regression Validation Metrics:', lr_metrics)

## Section 8: Random Forest

Train a non-linear ensemble tree model with class-weight balancing.

In [9]:
from sklearn.ensemble import RandomForestClassifier

rf_pipe = Pipeline([
    ('preprocessor', build_preprocessor(num_cols, cat_cols, scale_numeric=False)),
    ('classifier', RandomForestClassifier(n_estimators=config.rf_n_estimators, max_depth=config.rf_max_depth, class_weight='balanced', random_state=config.random_seed, n_jobs=-1)),
])
rf_pipe.fit(X_train, y_train)
rf_probs = rf_pipe.predict_proba(X_val)[:, 1]
rf_metrics = calculate_classification_metrics(y_val, rf_probs, threshold=0.50)
print('Random Forest Validation Metrics:', rf_metrics)

## Section 9: XGBoost

Train gradient-boosted decision trees using `scale_pos_weight` to address target class imbalance.

In [10]:
from xgboost import XGBClassifier

scale_pos_weight = float((y_train == 0).sum() / max((y_train == 1).sum(), 1))
xgb_pipe = Pipeline([
    ('preprocessor', build_preprocessor(num_cols, cat_cols, scale_numeric=False)),
    ('classifier', XGBClassifier(n_estimators=config.xgb_n_estimators, max_depth=config.xgb_max_depth, learning_rate=config.xgb_learning_rate, scale_pos_weight=scale_pos_weight, random_state=config.random_seed, eval_metric='logloss', n_jobs=-1)),
])
xgb_pipe.fit(X_train, y_train)
xgb_probs = xgb_pipe.predict_proba(X_val)[:, 1]
xgb_metrics = calculate_classification_metrics(y_val, xgb_probs, threshold=0.50)
print('XGBoost Validation Metrics:', xgb_metrics)

## Section 10: Model Comparison

Objectively benchmark all candidate models on the validation set across PR-AUC, ROC-AUC, F1, and Brier score.

In [11]:
models_val = {
    'Majority Baseline': dummy_metrics,
    'Logistic Regression': lr_metrics,
    'Random Forest': rf_metrics,
    'XGBoost': xgb_metrics,
}
comparison_df = compare_models(models_val)
display(comparison_df)

## Section 11: Threshold Sensitivity Analysis

Analyze precision, recall, and F1 trade-offs across candidate classification thresholds on the validation set.

In [12]:
thresh_df = evaluate_thresholds(y_val, lr_probs, thresholds=config.candidate_thresholds)
display(thresh_df)

# Select threshold with best validation F1
best_thresh = float(thresh_df.loc[thresh_df['f1'].idxmax()]['threshold'])
print(f'Selected Operational Threshold: {best_thresh:.2f}')

## Section 12: Probability Calibration

Analyze probability calibration reliability diagrams and Brier score loss.

In [13]:
calib_dict = compute_calibration_curve(y_val, lr_probs, n_bins=5)
print('Calibration reliability:', calib_dict)
print(f'Validation Brier Score: {calib_dict["brier_score"]:.4f}')

## Section 13: Feature Importance

Extract ranked feature importance using human-readable transformed feature names.

In [14]:
imp_df = extract_feature_importances(lr_pipe, num_cols, cat_cols)
print('Top 10 Feature Weights:')
display(imp_df.head(10))

## Section 14: SHAP Explainability

Compute SHAP values using a representative sample of training data.

In [15]:
shap_status, shap_imp = compute_shap_explainability(
    lr_pipe,
    X_train,
    num_cols,
    cat_cols,
    save_path=config.figures_dir / 'shap_summary.png',
)
print(f'SHAP Explainability Status: {shap_status}')
if shap_imp is not None:
    display(shap_imp.head(10))

## Section 15: Final Out-of-Time Test Evaluation

Evaluate the winning model once on the untouched out-of-time test partition at the selected threshold.

In [16]:
test_probs = lr_pipe.predict_proba(X_test)[:, 1]
final_test_metrics = calculate_classification_metrics(y_test, test_probs, threshold=best_thresh)
print('Final Unbiased Test Metrics (Evaluated ONCE):')
for k, v in final_test_metrics.items():
    print(f'  {k}: {v}')

## Section 16: Save Final Model & Serialization

Serialize the trained pipeline to `models/delay_model.joblib` and metadata to `models/model_metadata.json`.

In [17]:
meta_dict = {
    'model_name': 'Logistic Regression',
    'selected_threshold': best_thresh,
    'test_metrics': final_test_metrics,
    'dataset_limitation': 'Development sample evaluation (481 flights)',
}
model_p, meta_p = save_serialized_model(lr_pipe, meta_dict, config.models_dir)
print(f'Model pipeline saved to: {model_p}')
print(f'Metadata saved to: {meta_p}')